In [ ]:
import pandas as pd
import numpy as np

# Load your cleaned results table
results = pd.read_csv('..\\ACL_9th-F1\\F1_clean\\results_clean.csv')

print("--- PRE-MERGE AUDIT ---")

# 1. Grain Check: Is it strictly one row per driver, per race?
duplicates = results.duplicated(subset=['raceId', 'driverId'], keep=False)
num_dupes = duplicates.sum()
print(f"1. Grain Check: {num_dupes} duplicate (raceId, driverId) rows found.")
if num_dupes == 0:
    print("   ✅ PASS: Grain is correct.")
else:
    print("   ❌ FAIL: You must drop duplicates before merging.")

# 2. Sentinel Check: Are there any '\N' strings left?
has_N = (results == '\\N').any().any()
if not has_N:
    print("   ✅ PASS: No '\\N' sentinels found.")
else:
    print("   ❌ FAIL: '\\N' strings still exist. Replace with np.nan.")

# 3. Target Creation: Create the podium target now (Pre-merge)
print(f"3. Target 'podium' created. Baseline podium rate: {results['podium'].mean():.1%}")

## Feature Engineering Code

In [ ]:
nationality_to_country = {
    'British': 'UK', 'German': 'Germany', 'Spanish': 'Spain', 'Finnish': 'Finland',
    'Japanese': 'Japan', 'French': 'France', 'Polish': 'Poland', 'Brazilian': 'Brazil',
    'Italian': 'Italy', 'Australian': 'Australia', 'Austrian': 'Austria', 'American': 'USA',
    'Dutch': 'Netherlands', 'Colombian': 'Colombia', 'Portuguese': 'Portugal', 
    'Canadian': 'Canada', 'Indian': 'India', 'Hungarian': 'Hungary', 'Irish': 'Ireland',
    'Danish': 'Denmark', 'Argentine': 'Argentina', 'Belgian': 'Belgium', 
    'South African': 'South Africa', 'Swiss': 'Switzerland', 'Swedish': 'Sweden',
    'Mexican': 'Mexico', 'Russian': 'Russia', 'Chinese': 'China', 
    'Monegasque': 'Monaco', 'New Zealander': 'New Zealand'
}

In [ ]:
import pandas as pd
import numpy as np

# ============================================================================
# 1. LOAD CLEANED DATA 
# ============================================================================
base_path = '..\\ACL_9th-F1\\F1_clean\\'

results = pd.read_csv(base_path + 'results_clean.csv')
races = pd.read_csv(base_path + 'races_clean.csv')
driver_standings = pd.read_csv(base_path + 'driver_standings_clean.csv')
qualifying = pd.read_csv('..\\ACL_9th-F1\\F1\\' + 'qualifying.csv')
drivers = pd.read_csv(base_path + 'drivers_clean.csv')
circuits = pd.read_csv(base_path + 'circuits_clean.csv')
constructors = pd.read_csv(base_path + 'constructors_clean.csv')


# =============================================================================
# 2. DRIVER FORM FEATURES (Carries over between seasons)
# =============================================================================
# Merge results with races to sort chronologically
driver_form = results[['raceId', 'driverId', 'race_position_order', 'podium']].copy()
driver_form = driver_form.merge(races[['raceId', 'race_year', 'race_round']], on='raceId')

# Sort chronologically by driver
driver_form = driver_form.sort_values(by=['driverId', 'race_year', 'race_round'])

# A) Previous Position & Previous Podium (Grouped ONLY by driverId)
driver_form['driver_prev_position'] = driver_form.groupby('driverId')['race_position_order'].shift(1)
driver_form['driver_prev_podium'] = driver_form.groupby('driverId')['podium'].shift(1).fillna(0)

# B) Average Position in Last 5 Matches (Excludes current race)
driver_form['driver_avg_pos_last_5'] = (
    driver_form.groupby('driverId')['race_position_order']
    .shift(1)
    .groupby(driver_form['driverId'])
    .rolling(window=5, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
)

# Isolate columns for final merge
driver_form = driver_form[['raceId', 'driverId', 'driver_prev_position', 'driver_prev_podium', 'driver_avg_pos_last_5']]

# =============================================================================
# 3. DRIVER STANDINGS FEATURES (Resets to 0 every season)
# =============================================================================
# Merge standings with races to sort chronologically
ds = driver_standings.merge(races[['raceId', 'race_year', 'race_round']], on='raceId')
ds = ds.sort_values(by=['driverId', 'race_year', 'race_round'])

# A) Safe Pre-Race Points and Wins (Grouped by driverId AND race_year)
# Note: Ensure 'driver_points_cumm' and 'driver_wins_cumm' match your actual column names
ds['pre_race_points'] = ds.groupby(['driverId', 'race_year'])['driver_points_cumm'].shift(1).fillna(0)
ds['pre_race_wins'] = ds.groupby(['driverId', 'race_year'])['driver_wins_cumm'].shift(1).fillna(0)

# B) Did they win the last race? (Difference in season win total)
ds['won_previous_race'] = ds.groupby(['driverId', 'race_year'])['pre_race_wins'].diff().fillna(0)

# Isolate columns for final merge
ds = ds[['raceId', 'driverId', 'pre_race_points', 'pre_race_wins', 'won_previous_race']]

# =============================================================================
# 4. BUILD THE BASE TRAINING TABLE (Strict Leakage Control)
# =============================================================================
# Start with base results, keeping ONLY allowed pre-race info and the target
final_df = results[['raceId', 'driverId', 'constructorId', 'grid', 'podium']].copy()

# Hop 1: Race Context
final_df = final_df.merge(races[['raceId', 'race_year', 'race_round', 'circuitId']], on='raceId', how='left')

# Hop 2: Qualifying Performance 
final_df = final_df.merge(qualifying[['raceId', 'driverId', 'position']], on=['raceId', 'driverId'], how='left')
final_df.rename(columns={'position': 'quali_position'}, inplace=True)

# Hop 3: Driver Details & Circuit Details
# Assuming drivers table has 'nationality' and circuits table has 'country'
final_df = final_df.merge(drivers[['driverId', 'drivers_nationality']], on='driverId', how='left')
final_df.rename(columns={'drivers_nationality': 'driver_nationality'}, inplace=True)

final_df = final_df.merge(circuits[['circuitId', 'country']], on='circuitId', how='left')
final_df.rename(columns={'country': 'circuit_country'}, inplace=True)

# Hop 4: Constructor Details (The Car)
final_df = final_df.merge(constructors[['constructorId', 'name']], on='constructorId', how='left')
final_df.rename(columns={'name': 'constructor_name'}, inplace=True)

# Hop 5: Add Engineered Form & Standings
final_df = final_df.merge(driver_form, on=['raceId', 'driverId'], how='left')
final_df = final_df.merge(ds, on=['raceId', 'driverId'], how='left')

# =============================================================================
# 5. HOME RACE FEATURE ENGINEERING
# =============================================================================


# Map nationality to country string
final_df['driver_home_country'] = final_df['driver_nationality'].map(nationality_to_country).fillna(final_df['driver_nationality'])

# Standardize USA
#final_df['circuit_country'] = final_df['circuit_country'].replace({'United States': 'USA'})

# Create Boolean Feature (1 if match, 0 if not)
final_df['is_home_race'] = (final_df['driver_home_country'] == final_df['circuit_country']).astype(int)

# Clean up temp columns
final_df = final_df.drop(columns=['driver_home_country'])

# =============================================================================
# 6. FINAL CLEANUP & EXPORT
# =============================================================================
# Fill NaNs for drivers making their debut
final_df['pre_race_points'] = final_df['pre_race_points'].fillna(0)
final_df['pre_race_wins'] = final_df['pre_race_wins'].fillna(0)
final_df['won_previous_race'] = final_df['won_previous_race'].fillna(0)
final_df['driver_prev_podium'] = final_df['driver_prev_podium'].fillna(0)

# Note: driver_prev_position and driver_avg_pos_last_5 will naturally be NaN for debut races.
# You can impute them later during standard scaling/imputing (e.g., filling with 20).

final_df.to_csv(base_path + 'FINAL_training_table.csv', index=False)

print(f"Final Training Table built successfully! Shape: {final_df.shape}")
print("Columns:", final_df.columns.tolist())

In [ ]:
qualifying_original= pd.read_csv('..\\ACL_9th-F1\\F1\\qualifying.csv')

In [ ]:
qualifying_clean = pd.read_csv('..\\ACL_9th-F1\\F1_clean\\qualifying_clean_with_qs&pos.csv')

In [ ]:
constructors_clean = pd.read_csv('..\\ACL_9th-F1\\F1_clean\\constructors_clean.csv')

In [ ]:
constructors_original= pd.read_csv('..\\ACL_9th-F1\\F1\\constructors.csv')

In [ ]:
constructors_clean.info()

In [ ]:
constructors_original.info()

In [ ]:
final_df.info()

In [ ]:
results.info()